# UK Cost of Living & CPI Analysis

This notebook presents a cleaned analyst workflow for exploring cost-of-living indicators and building baseline predictive models. It is designed to run from the GitHub repository root.

## 1. Project Objectives

- Analyse cost-of-living indicators across countries and UK-related datasets.
- Identify relationships between rent, groceries, restaurant prices, purchasing power, CPI, and housing indicators.
- Produce clean charts and model outputs suitable for a portfolio project.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from config import DATA_RAW, DATA_PROCESSED, FIGURES_DIR, MODELS_DIR
from data_preprocessing import build_processed_datasets
from modeling import train_models

## 2. Data Preparation

Place all raw CSV files in `data/raw/`, then run the preprocessing pipeline below. The script standardises column names, converts numeric text values, handles duplicates, and writes cleaned files to `data/processed/`.

In [ ]:
build_processed_datasets()

In [ ]:
cost_df = pd.read_csv(DATA_PROCESSED / "cost_of_living_clean.csv")
cost_df.head()

## 3. Data Quality Check

In [ ]:
print("Rows:", cost_df.shape[0])
print("Columns:", cost_df.shape[1])
display(cost_df.info())

In [ ]:
missing_summary = cost_df.isna().sum().sort_values(ascending=False)
missing_summary[missing_summary > 0].head(20)

## 4. Exploratory Data Analysis

In [ ]:
numeric_df = cost_df.select_dtypes(include="number")
numeric_df.describe().T

### 4.1 Top Countries by Cost of Living Index

In [ ]:
if {"country", "cost_of_living_index"}.issubset(cost_df.columns):
    top_10 = cost_df.sort_values("cost_of_living_index", ascending=False).head(10)
    plt.figure(figsize=(10, 6))
    plt.barh(top_10["country"], top_10["cost_of_living_index"])
    plt.title("Top 10 Countries by Cost of Living Index")
    plt.xlabel("Cost of Living Index")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("Expected columns not found.")

### 4.2 Lowest Countries by Cost of Living Index

In [ ]:
if {"country", "cost_of_living_index"}.issubset(cost_df.columns):
    bottom_10 = cost_df.sort_values("cost_of_living_index", ascending=True).head(10)
    plt.figure(figsize=(10, 6))
    plt.barh(bottom_10["country"], bottom_10["cost_of_living_index"])
    plt.title("Bottom 10 Countries by Cost of Living Index")
    plt.xlabel("Cost of Living Index")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("Expected columns not found.")

### 4.3 Correlation Between Cost Indicators

In [ ]:
plt.figure(figsize=(11, 8))
sns.heatmap(numeric_df.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Matrix: Cost-of-Living Indicators")
plt.tight_layout()
plt.show()

### 4.4 Average Component Breakdown

In [ ]:
component_cols = ["rent_index", "groceries_index", "restaurant_price_index", "local_purchasing_power_index"]
available = [col for col in component_cols if col in cost_df.columns]
if available:
    avg_components = cost_df[available].mean().sort_values(ascending=False)
    plt.figure(figsize=(9, 5))
    plt.bar(avg_components.index, avg_components.values)
    plt.title("Average Cost-of-Living Component Scores")
    plt.ylabel("Average Index Value")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No expected component columns found.")

## 5. Modelling

The modelling section predicts `cost_of_living_index` using the other numeric indicators. This gives a clean baseline comparison between linear and tree-based regression models.

In [ ]:
results_df = train_models(target="cost_of_living_index")
results_df

## 6. Findings

Update this section after running the notebook with your actual outputs.

Suggested analyst summary:

- The cost-of-living index is strongly related to rent, groceries, and restaurant price indicators.
- Purchasing power should be interpreted separately because a high value can indicate better affordability rather than higher cost.
- Tree-based models may capture non-linear relationships better than linear regression, but the final model should be selected using RMSE and interpretability.

## 7. Limitations and Next Steps

- Add official source links for each dataset.
- Expand the modelling section with time-series validation if the target is chronological CPI or housing data.
- Build a Power BI, Tableau, or Streamlit dashboard.
- Add unit tests and automated checks for data quality.